# Anatomy of the consolidated tape

This notebook reconstructs the cross-venue tape from raw trades for a short window so you can *see* what the construction does: the k-way merge, the per-second robust (median + MAD) outlier filter, and the roll-up into 1-second bars with provenance. It then sanity-checks the resulting `agg` bars against the per-venue series.

Run `cda-build-dataset` first. Plots use `matplotlib` (`pip install matplotlib`).

In [ ]:
import pandas as pd
from cryptodata import get_trades
from cryptodata.core.consolidated import merge_trades, build_agg_bars
from cryptodata.core.symbols import venue_bitmask_index
from cryptodata.sources.registry import all_sources, SPOT_VENUES
from cryptodata.storage.duckdb_views import connect

SYMBOL = 'BTC-USD'
with connect() as con:
    lo, hi = con.execute('SELECT MIN(ts_ns), MAX(ts_ns) FROM trades WHERE symbol = ?', [SYMBOL]).fetchone()
# take a 3-minute slice from the middle of what we have
mid = (lo + hi) // 2
w0, w1 = mid - 90 * 1_000_000_000, mid + 90 * 1_000_000_000
raw = get_trades(SYMBOL, pd.Timestamp(w0, unit='ns', tz='UTC'), pd.Timestamp(w1, unit='ns', tz='UTC'), venues='all')
print(f'{len(raw)} trades over {len(raw["venue"].unique())} venues:'); print(raw['venue'].value_counts())

### 1. k-way merge → one stream ordered by exchange timestamp

In [ ]:
per_venue = {v: g.reset_index().assign(ts_ns=lambda d: d['ts'].astype('int64'))[['ts_ns', 'price', 'size', 'venue']].assign(recv_ns=lambda d: d['ts_ns']).to_dict('records')
             for v, g in raw.groupby('venue') if v in SPOT_VENUES}
for v in per_venue:
    per_venue[v].sort(key=lambda r: r['ts_ns'])
merged = merge_trades(per_venue)
print(f'consolidated stream: {len(merged)} trades; first 5:')
pd.DataFrame(merged[:5])

### 2. Build the agg bars with diagnostics — see what the outlier filter dropped

In [ ]:
idx = venue_bitmask_index(all_sources())
agg_bars, diags = build_agg_bars(merged, symbol=SYMBOL, venues_index=idx, mad_k=5.0, min_venues_for_filter=3, return_diagnostics=True)
diag = pd.DataFrame(diags)
print(f'{len(agg_bars)} agg bars; {diag["n_dropped"].sum()} trades dropped by the robust filter '
      f'({100*diag["n_dropped"].sum()/max(diag["n_raw"].sum(),1):.3f}% of trades)')
print('seconds where the filter actually fired:'); print(diag[diag['n_dropped'] > 0].head())

### 3. Sanity check: the agg close must sit between the per-venue extremes

In [ ]:
agg = pd.DataFrame(agg_bars).set_index('ts_ns')['close']
venue_close = {}
for v, rows in per_venue.items():
    s = pd.Series({(r['ts_ns'] // 1_000_000_000) * 1_000_000_000: r['price'] for r in rows})  # last trade per second
    venue_close[v] = s
vc = pd.concat(venue_close, axis=1)
common = agg.index.intersection(vc.index)
lo_v, hi_v = vc.loc[common].min(axis=1), vc.loc[common].max(axis=1)
breaches = ((agg.loc[common] < lo_v - 1e-6) | (agg.loc[common] > hi_v + 1e-6)).sum()
print(f'{len(common)} seconds compared; {breaches} where agg fell outside [min_venue, max_venue] (should be 0)')

In [ ]:
try:
    import matplotlib.pyplot as plt
    fig, ax = plt.subplots(figsize=(12, 5))
    for v in vc.columns:
        ax.plot(pd.to_datetime(vc.index, unit='ns', utc=True), vc[v], lw=0.6, alpha=0.6, label=v)
    ax.plot(pd.to_datetime(agg.index, unit='ns', utc=True), agg.values, lw=1.4, color='k', label='agg (consolidated)')
    ax.set_title(f'{SYMBOL} — consolidated tape vs. per-venue last-trade-per-second'); ax.legend(); plt.tight_layout(); plt.show()
except ImportError:
    print('install matplotlib for the plot')